In [71]:
from importlib import import_module

named_libs = [('pandas', 'pd'), ('numpy', 'np')] # (library_name, shorthand)
for (name, short) in named_libs:
    try:
        lib = import_module(name)
    except:
        print(sys.exc_info())
    else:
        globals()[short] = lib  

libnames = ['configparser', 'pyexasol', 'os']
for libname in libnames:
    try:
        lib = import_module(libname)
    except:
        print(sys.exc_info())
    else:
        globals()[libname] = lib

def step_1(config_path, sql_path1, sql_path2):   
    #Location of the ini file
    config = configparser.ConfigParser()
    config.read(config_path)
    dsn=config['exasolPROD']['dsn']
    user=config['exasolPROD']['user']
    pwd=config['exasolPROD']['pwd']
    schema=config['exasolPROD']['schema']
    # Exasol connection
    connect = pyexasol.connect(dsn=dsn, user=user, password=pwd, schema=schema)
    import_query_open = open(sql_path1, 'r')
    import_query_read = import_query_open.read()
    df = connect.export_to_pandas(import_query_read)
    
    import_query_open = open(sql_path2, 'r')
    import_query_read = import_query_open.read()
    df_cc = connect.export_to_pandas(import_query_read)
    return df, df_cc

def step_2(df):
    #This sample file does not include hotel_category_id and hotel_city_id. But we can include those two columns to ease the following steps.

    def rm500(a):
        if a==500:
            return np.nan
        else:
            return a

    df['EX_LB'] = [rm500(x) for x in df['EX_LB']]
    df['EX_UB'] = [rm500(x) for x in df['EX_UB']]

    #More patterns of outlier can be discovered. 

    def get_extreme(a):
        lb = np.percentile(a.dropna(),1)
        ub = np.percentile(a.dropna(),99)
        return [lb,ub]
    def rm_extreme(a):
        a_lb = np.percentile(a.dropna(),1)
        a_ub = np.percentile(a.dropna(),99)
        a[a<a_lb] = np.nan
        a[a>a_ub] = np.nan
        return a

    df_rm_extreme = df[['CR_LB','BK_LB','EX_LB','FD_LB','GAIN_LB','BID_LB','CR_UB','BK_UB','EX_UB','FD_UB','GAIN_UB','BID_UB']]
    df_no_extreme = df_rm_extreme.apply(rm_extreme,axis=0)
    #df['Median_LB'] = get_median(df['CR_LB'],df['BK_LB'],df['EX_LB'])   
    #df['Median_UB'] = get_median(df['CR_UB'],df['BK_UB'],df['EX_UB'])      
    #df['Min_CRO_BKG'] = get_min(df['CR_LB'],df['BK_LB'])
    #df_lb = df[['CR_LB','BK_LB','EX_LB','HS_LB','FD_LB','GAIN_LB','BID_LB']]
    df_lb = df_no_extreme[['CR_LB','BK_LB','EX_LB','FD_LB','GAIN_LB','BID_LB']]
    med_lb = df_lb.apply(lambda x: np.nanmedian(x),axis=1)
    df['Median_LB'] = med_lb
    df_ub = df_no_extreme[['CR_UB','BK_UB','EX_UB','FD_UB','GAIN_UB','BID_UB']]
    #f_ub = df[['CR_UB','BK_UB','EX_UB','HS_LB','FD_UB','GAIN_UB','BID_UB']]
    med_ub = df_ub.apply(lambda x: np.nanmedian(x),axis=1)
    df['Median_UB'] = med_ub
    df['Diff_LB_UB'] = df['Median_UB']/df['Median_LB']
    adj_ub = []
    adj_lb = []
    multiplier_up_lw = 2.5
    m1 = (multiplier_up_lw-1)/(1+multiplier_up_lw)
    for i in range(len(df)):
        if (df.loc[i,'Median_UB']/df.loc[i,'Median_LB']) <multiplier_up_lw:
            mid = (df.loc[i,'Median_LB']+ df.loc[i,'Median_UB'])/2
            adj_lb.append(mid*(1-m1))
            adj_ub.append(mid*(1+m1))
        else:
            adj_lb.append(df.loc[i,'Median_LB'])
            adj_ub.append(df.loc[i,'Median_UB'])
    df['Adj_LB']=adj_lb
    df['Adj_UB']=adj_ub
    df_1 = df[(df['Adj_LB'].isna()==False) & (df['Diff_LB_UB']<=4)]
    #df_11 = df_1[['HOTEL_ID', 'HOTEL_NAME', 'HOTEL_CATEGORY_SHORT_DESC',
    #       'HOTEL_CITY_NAME', 'COUNTRY_NAME','Adj_LB', 'Adj_UB']]
    df_11 = df_1[['HOTEL_ID', 'HOTEL_CATEGORY_ID', 'HOTEL_CITY_ID', 'Adj_LB', 'Adj_UB']]
    df_11.rename(columns ={'Adj_LB':'HOTEL_RATE_LOWER_BOUND', 'Adj_UB': 'HOTEL_RATE_UPPER_BOUND'}, inplace = True, errors="raise")
    
    return df_11

def step_3(df_cc):
    city_cat = df_cc
    for i in range(len(city_cat)):
        if city_cat.loc[i,'BK_UB_CC']/city_cat.loc[i,'BK_LB_CC']<=1.05:
            city_cat.loc[i,'BK_UB_CC'] = np.nan
            city_cat.loc[i,'BK_LB_CC'] = np.nan         
    for i in range(len(city_cat)):
        if city_cat.loc[i,'HS_UB_CC']/city_cat.loc[i,'HS_LB_CC']<=1.05:
            city_cat.loc[i,'HS_UB_CC'] = np.nan
            city_cat.loc[i,'HS_LB_CC'] = np.nan   
    city_cat_ub = city_cat[['BK_UB_CC','CR_UB_CC','EX_UB_CC','HS_UB_CC',\
                            'FD_UB_CC','GAIN_UB_CC','BID_UB_CC']]
    city_cat_med_ub = city_cat_ub.apply(lambda x:np.nanmedian(x),axis=1)
    city_cat_lb = city_cat[['BK_LB_CC','CR_LB_CC','EX_LB_CC','HS_LB_CC',\
                            'FD_LB_CC','GAIN_LB_CC','BID_LB_CC']]
    city_cat_med_lb = city_cat_lb.apply(lambda x:np.nanmedian(x),axis=1)
    city_cat_min_lb = city_cat_lb.apply(lambda x:np.nanmin(x),axis=1)
    city_cat_max_ub = city_cat_ub.apply(lambda x:np.nanmax(x),axis=1) 
    city_cat_bounds =  city_cat[['HOTEL_CITY_ID', 'HOTEL_CATEGORY_ID']] 
    city_cat_bounds['MED_LB_CC'] = city_cat_med_lb
    city_cat_bounds['MED_UB_CC'] = city_cat_med_ub
    city_cat_bounds['MIN_LB_CC'] = city_cat_min_lb
    city_cat_bounds['MAX_UB_CC'] = city_cat_max_ub 
    adj_lb = []
    adj_ub = []
    multiplier = 3
    m1 = (multiplier-1)/(multiplier+1)
    for i in range(len(city_cat_bounds)):
        if (city_cat_bounds.loc[i,'MED_UB_CC']/city_cat_bounds.loc[i,'MED_LB_CC'])<multiplier:
            mid = (city_cat_bounds.loc[i,'MED_UB_CC']+city_cat_bounds.loc[i,'MED_LB_CC'])/2
            adj_lb.append(mid*(1-m1))
            adj_ub.append(mid*(1+m1))
        else:
            adj_lb.append(city_cat_bounds.loc[i,'MED_LB_CC'])
            adj_ub.append(city_cat_bounds.loc[i,'MED_UB_CC'])
    city_cat_bounds['Adj_LB_CC'] = adj_lb
    city_cat_bounds['Adj_UB_CC'] = adj_ub
    city_cat_bounds['Diff_Bounds']= city_cat_bounds['Adj_UB_CC']/city_cat_bounds['Adj_LB_CC']
    city_cat_bounds = city_cat_bounds[(city_cat_bounds['Adj_LB_CC'].isna()==False) & (city_cat_bounds['Diff_Bounds']<6)]  
    bounds_city_cat = city_cat_bounds[['HOTEL_CITY_ID', 'HOTEL_CATEGORY_ID','Adj_LB_CC', 'Adj_UB_CC']]
    city_bounds= pd.pivot_table(bounds_city_cat,
                                  index=['HOTEL_CITY_ID'], 
                                  values = ['Adj_LB_CC','Adj_UB_CC'],
                                  aggfunc ={'Adj_LB_CC':'min',
                                            'Adj_UB_CC':'max'}).reset_index()
    city_bounds.columns=['HOTEL_CITY_ID', 'Adj_LB_CITY', 'Adj_UB_CITY']    
    city_category_city_bounds = pd.merge(bounds_city_cat,city_bounds,how="inner",on=['HOTEL_CITY_ID'])
    city_category_city_bounds.rename(columns={"Adj_LB_CC": "HOTEL_CITY_CATEGORY_RATE_LOWER_BOUND", "Adj_UB_CC": "HOTEL_CITY_CATEGORY_RATE_UPPER_BOUND", "Adj_LB_CITY": "HOTEL_CITY_RATE_LOWER_BOUND", "Adj_UB_CITY": "HOTEL_CITY_RATE_UPPER_BOUND"}, errors="raise", inplace = True)
    return city_category_city_bounds

In [72]:
## Step - 1 function call:
config_path = 'C:\\Users\\USER\\.spyder-py3\\ExasolPROD.ini'
sql_path1 = 'C:/Users/USER/Documents/outlier_detection_step_by_step_implementation/step1_hotel_bounds_modified.sql'
sql_path2 = 'C:/Users/USER/Documents/outlier_detection_step_by_step_implementation/step3_cc_bounds_modified.sql'
df, df_cc = step_1(config_path, sql_path1, sql_path2)

## Step - 2 function call with Step - 1 results:
hotel_level_bounds = step_2(df)

## Step - 3 function call:
city_category_city_bounds = step_3(df_cc)

## Step - 5:
df_wcc = pd.merge(hotel_level_bounds, city_category_city_bounds, how='left', on = ['HOTEL_CATEGORY_ID', 'HOTEL_CITY_ID'])

In [65]:
#Location of the ini file
config = configparser.ConfigParser()
config.read('C:\\Users\\USER\\.spyder-py3\\ExasolDET.ini')
dsn=config['exasolDET']['dsn']
user=config['exasolDET']['user']
pwd=config['exasolDET']['pwd']
schema=config['exasolDET']['schema']
# Exasol connection
connect = pyexasol.connect(dsn=dsn, user=user, password=pwd, schema=schema)
import_query_location = sql_path3 
import_query_open = open(import_query_location, 'r')
import_query_read = import_query_open.read()
connect.execute("TRUNCATE TABLE TEMP.REL_OUTLIER_RANGE")
connect.import_from_pandas(df_wcc, table = ('TEMP','REL_OUTLIER_RANGE'))

In [73]:
df.head() 

In [74]:
df_cc.head() 

In [75]:
hotel_level_bounds.head() 

In [76]:
city_category_city_bounds.head() 

In [77]:
df_wcc.head() 

In [79]:
df.shape 

In [80]:
df_cc.shape

In [81]:
hotel_level_bounds.shape 

In [82]:
city_category_city_bounds.shape

In [83]:
df_wcc.shape